# Artefactual Package Demo: Hallucination Detection with EPR 

This notebook demonstrates the `artefactual` package for scoring LLM outputs, specifically focusing on hallucination detection using entropy-based methods. Here we will use EPR (Entropy Production Rate), which computes entropy at each token and averages it across the entire sequence.

We explore two examples:
1.  **General Knowledge Question**: 

    *"What is the capital city of France?"* → `"Paris."` (expected high certainty, low entropy expected)

2.  **Hallucination Trigger**: Asking about the first author of our paper (Charles Moslonka) to observe how the model hallucinates biographical details.

    *"Who is Charles Moslonka?"* → a fabricated biography (expected high uncertainty, high entropy expected)

We will use:
* **JSON fixture (open_ai_responses_top15.json):** two mock OpenAI Responses API outputs, carrying 15 top logprobs per token -- the rank count the shipped calibrations were fit at.
* **Published detector:** named by its own Hugging Face repository (`artefactory/epr-ministral`), fetched on first use and cached. A path to a local `.skops` file is accepted the same way.
* **EPR:** scorer from the artefactual package via the scikit-learn pipeline API.



In [1]:
# On Colab, uncomment to install the package and fetch the files this notebook reads.
# !pip install -q artefactual
# !wget -q https://raw.githubusercontent.com/artefactory/artefactual/main/docs/examples/open_ai_responses_top15.json

In [2]:
import json
from pathlib import Path

from artefactual.scoring import EPR, BaseDetector

In [3]:
# The Hugging Face repository holding the published EPR detector for the model
# that produced these responses. A path to your own `.skops` file works too.
DETECTOR = "artefactory/epr-ministral"
DATA_PATH = "open_ai_responses_top15.json"

# The rank count the detector was trained at, which the fixture also carries.
# A response narrower than K is refused rather than padded: the missing ranks
# are unfetched, not absent, so filling them would understate the entropy.
K = 15

## Load Example Responses

The fixture contains two responses to illustrate the contrast between a certain and a uncertain answer.

In [4]:
fixture_path = Path(DATA_PATH)
with fixture_path.open(encoding="utf-8") as f:
    data = json.load(f)

responses = data["responses"]
print(f"Loaded {len(responses)} responses")

Loaded 2 responses


## Build the EPR Pipeline

The detector is fetched from the Hub on first use, then cached. `EPR()` always requires a repository id or a path.

In [5]:
detector = EPR.from_pretrained(DETECTOR, k=K)

## Sequence-Level Scoring

`predict_proba(response)` returns an array of shape `(n_sequences, 2)`.
Column 1 is the hallucination probability, higher means the model was more uncertain.

In [6]:
for resp in responses:
    prompt = resp.get("metadata", {}).get("prompt", "")
    text = resp["output"][0]["content"][0]["text"]
    score = detector.predict_proba(resp)[:, 1][0]
    print(f"Prompt : {prompt}")
    print(f"Answer : {text}")
    print(f"EPR score: {score:.2f}")
    print()

Prompt : What is the capital city of France? Please answer briefly.
Answer : Paris.
EPR score: 0.05

Prompt : Who is Charles Moslonka ? Where was he born ? Please answer in two sentences.
Answer : Charles Moslonka is a French singer born in Lyon in 1985.
EPR score: 1.00



## Token Level Scoring

`predict_token_proba(response)` returns an array of shape `(n_sequences, max_tokens, 1)`.

* **`n_sequences`**: Response index.
* **`max_tokens`**: Token index within the sequence.
* **`1`**: The per-token scalar hallucination probability.


In [7]:
for resp in responses:
    prompt = resp.get("metadata", {}).get("prompt", "")
    token_scores = detector.predict_token_proba(resp)  # shape: (1, max_tokens, 1)
    scores = token_scores[0, :, 0]

    # Extract token strings from the response logprobs
    tokens = [t["token"] for t in resp["output"][0]["content"][0]["logprobs"]]

    print(f"Prompt: {prompt}")
    for token, score in zip(tokens, scores):
        print(f"  {token!r:13s} → {score:.2f}")
    print()

Prompt: What is the capital city of France? Please answer briefly.
  'Paris'       → 0.05
  '.'           → 0.05
  '</s>'        → 0.05

Prompt: Who is Charles Moslonka ? Where was he born ? Please answer in two sentences.
  'Charles'     → 0.05
  ' Moslonka'   → 0.05
  ' is'         → 1.00
  ' a'          → 1.00
  ' French'     → 1.00
  ' singer'     → 1.00
  ' born'       → 0.05
  ' in'         → 0.05
  ' Lyon'       → 1.00
  ' in 1985'    → 1.00
  '.'           → 1.00



## Your model is not one of the published detectors?

A detector reads one model's confidence, so the four published ones only score the four
models they were trained on. `EPR(k=15).fit(responses, y)` fits your
own, on that model's answers and a verdict on each.
